
# Part 3 – Optimization of Cargo Operations

This notebook solves **Part 3 only** of the airport project by building and solving a **linear optimization model** for Express Air's weekly cargo operations among airports **A, B, and C**.

The notebook is written to match the optimization methodology used in class:

1. **Define the decision variables**
2. **Write the objective function**
3. **Write the constraints**
4. **Classify the optimization problem**
5. **Use a computational solver**
6. **Interpret the managerial implications**

I also include a lot of comments in the code and Markdown explanations so it is easy to follow.



## 1. Problem setup and modeling idea

Express Air operates over the five weekdays: **Monday through Friday**.

Each day:

- new cargo arrives for each origin-destination pair,
- the company decides how much cargo to ship **loaded**,
- the company may also **reposition aircraft empty**,
- unused aircraft can **stay overnight** at the same airport,
- any cargo that is not shipped becomes **backlog** and carries over to the next operating day.

Because travel time is **one full day**, any aircraft that leaves airport $i$ on day $t$ becomes available at airport $j$ at the **beginning of the next day**.

The company wants a **repeatable weekly cycle**, so the state at the beginning of Monday must match what comes out of Friday's decisions.

### Important modeling assumptions used here

I make the following modeling choices, which are consistent with the project statement:

- The **total fleet size** is fixed at **1200 aircraft**.
- The **distribution** of those 1200 aircraft across A, B, and C on Monday morning is **not given**, so it is treated as something the optimization can choose endogenously as part of the best repeating cycle.
- Loaded cargo movement cost is ignored, because the prompt says loaded cargo must eventually be moved anyway, so that part is fixed and does not affect the optimization.
- The objective includes:
  - **empty repositioning cost**, and
  - **cargo holding cost** of **10 per aircraft-load per day**.
- Since the carrier **does not operate on weekends**, backlog carried from **Friday morning to Monday morning** is charged for **3 days** of holding in this notebook. I state that assumption explicitly because weekend holding is not spelled out in the prompt and should not be left hidden.

This produces a clean **linear program (LP)**.


In [3]:

# Standard imports
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Make tables display nicely in the notebook
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")



## 2. Input data from the project

The weekly cargo arrivals (in full aircraft loads) are:

- A→B: 100, 200, 100, 400, 300
- A→C: 50 each day
- B→A: 25 each day
- B→C: 25 each day
- C→A: 40 each day
- C→B: 400, 200, 300, 200, 400

The empty repositioning costs are:

- A↔B: 7
- A↔C: 3
- B↔C: 6

The total fleet size is **1200 aircraft**.


In [4]:

# -----------------------------
# Raw data from the project
# -----------------------------

days = ["Mon", "Tue", "Wed", "Thu", "Fri"]
airports = ["A", "B", "C"]

# Ordered list of all directed origin-destination pairs
ods = [(i, j) for i in airports for j in airports if i != j]

# Cargo arrivals by day and OD pair
demand = {
    ("Mon", "A", "B"): 100, ("Tue", "A", "B"): 200, ("Wed", "A", "B"): 100, ("Thu", "A", "B"): 400, ("Fri", "A", "B"): 300,
    ("Mon", "A", "C"):  50, ("Tue", "A", "C"):  50, ("Wed", "A", "C"):  50, ("Thu", "A", "C"):  50, ("Fri", "A", "C"):  50,
    ("Mon", "B", "A"):  25, ("Tue", "B", "A"):  25, ("Wed", "B", "A"):  25, ("Thu", "B", "A"):  25, ("Fri", "B", "A"):  25,
    ("Mon", "B", "C"):  25, ("Tue", "B", "C"):  25, ("Wed", "B", "C"):  25, ("Thu", "B", "C"):  25, ("Fri", "B", "C"):  25,
    ("Mon", "C", "A"):  40, ("Tue", "C", "A"):  40, ("Wed", "C", "A"):  40, ("Thu", "C", "A"):  40, ("Fri", "C", "A"):  40,
    ("Mon", "C", "B"): 400, ("Tue", "C", "B"): 200, ("Wed", "C", "B"): 300, ("Thu", "C", "B"): 200, ("Fri", "C", "B"): 400,
}

# Empty repositioning costs (symmetric in this project)
reposition_cost = {
    ("A", "B"): 7, ("B", "A"): 7,
    ("A", "C"): 3, ("C", "A"): 3,
    ("B", "C"): 6, ("C", "B"): 6,
}

# Cargo holding cost
holding_cost_per_day = 10

# Total fleet size
total_fleet = 1200

# Because the carrier does not operate on weekends,
# carrying cargo from Friday morning to Monday morning is modeled
# as 3 days of holding.
backlog_holding_weight = {
    "Mon": 3,  # Friday -> Monday
    "Tue": 1,  # Monday -> Tuesday
    "Wed": 1,  # Tuesday -> Wednesday
    "Thu": 1,  # Wednesday -> Thursday
    "Fri": 1,  # Thursday -> Friday
}

# Put the cargo table into a DataFrame for display
cargo_table = pd.DataFrame(
    {day: [demand[(day, o, d)] for (o, d) in ods] for day in days},
    index=[f"{o}->{d}" for (o, d) in ods]
)

cargo_table


,Mon,Tue,Wed,Thu,Fri
A->B,100,200,100,400,300
A->C,50,50,50,50,50
B->A,25,25,25,25,25
B->C,25,25,25,25,25
C->A,40,40,40,40,40
C->B,400,200,300,200,400



## 3. Decision variables

For each day $t$ and each ordered airport pair $(i,j)$ with $i \neq j$:

- $L_{t,i,j}$ = number of **loaded** aircraft moving from $i$ to $j$ on day $t$
- $E_{t,i,j}$ = number of **empty repositioning** aircraft moving from $i$ to $j$ on day $t$

For each day $t$ and airport $i$:

- $S_{t,i}$ = number of aircraft that **stay overnight** at airport $i$ after day $t$
- $A_{t,i}$ = number of aircraft **available at the beginning of day $t$** at airport $i$

For each day $t$ and ordered airport pair $(i,j)$:

- $B_{t,i,j}$ = amount of **backlog cargo** waiting at the **beginning of day $t$**

These are the only variables we need.

---

## 4. Objective function

We minimize total weekly cost:

$ \min \sum_{t,i,j} c_{i,j} E_{t,i,j} + \sum_{t,i,j} h_t \cdot 10 \cdot B_{t,i,j}
$

where:

- $c_{i,j}$ is the empty repositioning cost,
- $10$ is the per-day holding cost per aircraft-load of cargo,
- $h_t$ is the holding multiplier:
  - 1 for Tue, Wed, Thu, Fri backlogs,
  - 3 for Monday backlog, because that backlog represents cargo held over the weekend.

---

## 5. Constraints

### 5.1 Aircraft usage at each airport on each day

Every aircraft available at the start of day $t$ at airport $i$ must do exactly one of:

- carry cargo out of $i$,
- reposition empty out of $i$,
- stay at $i$.

So for each day $t$ and airport $i$,

$ A_{t,i} = S_{t,i} + \sum_{j \neq i} L_{t,i,j} + \sum_{j \neq i} E_{t,i,j}$

### 5.2 Aircraft transition to the next day

Any aircraft that flies into airport $i$ on day $t$, whether loaded or empty, is available there at the beginning of the next day. Also, aircraft that stay overnight remain there.

So for each day $t$ and airport $i$,

$ A_{t+1,i} = S_{t,i} + \sum_{j \neq i} L_{t,j,i} + \sum_{j \neq i} E_{t,j,i}$

For Friday, "next day" means **Monday**, which creates the weekly cycle.

### 5.3 Total fleet size

For each day $t$,

$ \sum_i A_{t,i} = 1200 $

### 5.4 Cargo flow balance

For each day $t$ and OD pair $(i,j)$,

$
B_{t+1,i,j} = B_{t,i,j} + D_{t,i,j} - L_{t,i,j}
$

where $D_{t,i,j}$ is the new cargo arriving that day.

This says:

> tomorrow's backlog = today's backlog + today's new arrivals - today's shipments

Again, Friday's next day is Monday.

### 5.5 Cannot ship more than available cargo

For each day $t$ and OD pair $(i,j)$,

$
L_{t,i,j} \le B_{t,i,j} + D_{t,i,j}
$

### 5.6 Nonnegativity

All variables are constrained to be nonnegative.

---

## 6. What type of optimization problem is this?

This is a **linear optimization problem** because:

- the objective function is a sum of constants times decision variables,
- every constraint is linear,
- there are no products of decision variables,
- there are no powers, logs, exponentials, or other nonlinear terms.

So this is exactly the kind of formulation that fits the linear-programming style discussed in class and in the homework examples.


In [5]:

# -------------------------------------------------------------
# Build a variable index so that we can write the LP in matrix form
# -------------------------------------------------------------

variables = []

# Loaded movement variables
for t in days:
    for (o, d) in ods:
        variables.append(("loaded", t, o, d))

# Empty repositioning variables
for t in days:
    for (o, d) in ods:
        variables.append(("empty", t, o, d))

# Stay variables
for t in days:
    for i in airports:
        variables.append(("stay", t, i))

# Backlog-at-start-of-day variables
for t in days:
    for (o, d) in ods:
        variables.append(("backlog", t, o, d))

# Aircraft available-at-start-of-day variables
for t in days:
    for i in airports:
        variables.append(("avail", t, i))

# Map each variable tuple to a column number in the LP
var_index = {var: k for k, var in enumerate(variables)}
n_vars = len(variables)

print(f"Number of decision variables in the LP: {n_vars}")


Number of decision variables in the LP: 120


In [6]:

# -------------------------------------------------------------
# Helper functions for building equality and inequality rows
# -------------------------------------------------------------

A_eq = []
b_eq = []

A_ub = []
b_ub = []

def add_eq(coeffs, rhs):
    """Add one equality constraint of the form row * x = rhs."""
    row = np.zeros(n_vars)
    for var, val in coeffs.items():
        row[var_index[var]] = val
    A_eq.append(row)
    b_eq.append(rhs)

def add_ub(coeffs, rhs):
    """Add one inequality constraint of the form row * x <= rhs."""
    row = np.zeros(n_vars)
    for var, val in coeffs.items():
        row[var_index[var]] = val
    A_ub.append(row)
    b_ub.append(rhs)

# Successor day mapping used to enforce the weekly cycle
next_day = {days[i]: days[(i + 1) % len(days)] for i in range(len(days))}
next_day


{'Mon': 'Tue', 'Tue': 'Wed', 'Wed': 'Thu', 'Thu': 'Fri', 'Fri': 'Mon'}

In [7]:

# -------------------------------------------------------------
# 1) Aircraft usage constraints:
#    available = stay + loaded departures + empty departures
# -------------------------------------------------------------
for t in days:
    for i in airports:
        coeffs = {
            ("avail", t, i): 1,
            ("stay", t, i): -1,
        }
        for j in airports:
            if j != i:
                coeffs[("loaded", t, i, j)] = -1
                coeffs[("empty",  t, i, j)] = -1
        add_eq(coeffs, 0)

# -------------------------------------------------------------
# 2) Aircraft transition constraints:
#    next day's availability = today's stay + today's incoming aircraft
# -------------------------------------------------------------
for t in days:
    t_next = next_day[t]
    for i in airports:
        coeffs = {
            ("avail", t_next, i): 1,
            ("stay",  t, i): -1,
        }
        for j in airports:
            if j != i:
                coeffs[("loaded", t, j, i)] = -1
                coeffs[("empty",  t, j, i)] = -1
        add_eq(coeffs, 0)

# -------------------------------------------------------------
# 3) Fleet size constraints:
#    total aircraft available each day = 1200
# -------------------------------------------------------------
for t in days:
    coeffs = {("avail", t, i): 1 for i in airports}
    add_eq(coeffs, total_fleet)

# -------------------------------------------------------------
# 4) Cargo flow-balance constraints:
#    next backlog = current backlog + arrivals - loaded shipments
# -------------------------------------------------------------
for t in days:
    t_next = next_day[t]
    for (o, d) in ods:
        coeffs = {
            ("backlog", t_next, o, d):  1,
            ("backlog", t,      o, d): -1,
            ("loaded",  t,      o, d):  1,
        }
        # move demand to the RHS
        add_eq(coeffs, demand[(t, o, d)])

# -------------------------------------------------------------
# 5) Cargo feasibility constraints:
#    loaded shipments cannot exceed available cargo
#    loaded <= backlog + today's arrivals
# -------------------------------------------------------------
for t in days:
    for (o, d) in ods:
        coeffs = {
            ("loaded",  t, o, d):  1,
            ("backlog", t, o, d): -1,
        }
        add_ub(coeffs, demand[(t, o, d)])

# Convert lists to arrays for scipy
A_eq = np.array(A_eq)
b_eq = np.array(b_eq)
A_ub = np.array(A_ub)
b_ub = np.array(b_ub)

print("Equality matrix shape:", A_eq.shape)
print("Inequality matrix shape:", A_ub.shape)


Equality matrix shape: (65, 120)
Inequality matrix shape: (30, 120)


In [8]:

# -------------------------------------------------------------
# Build the objective vector
# -------------------------------------------------------------
c = np.zeros(n_vars)

# Empty repositioning cost
for t in days:
    for (o, d) in ods:
        c[var_index[("empty", t, o, d)]] = reposition_cost[(o, d)]

# Backlog holding cost
# The Monday backlog corresponds to Friday -> Monday carryover,
# so it gets weighted by 3 days of holding cost.
for t in days:
    for (o, d) in ods:
        c[var_index[("backlog", t, o, d)]] = holding_cost_per_day * backlog_holding_weight[t]

# All variables are nonnegative
bounds = [(0, None)] * n_vars

# Solve with HiGHS via scipy
result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

print("Optimization success:", result.success)
print("Solver status:", result.status)
print("Solver message:", result.message)
print("Optimal weekly cost:", result.fun)


Optimization success: True
Solver status: 0
Solver message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Optimal weekly cost: 21725.0



## 7. Extracting the solution into readable tables

The solver returns one long vector of numbers. The next cell converts that vector back into:

- aircraft availability by day and airport,
- loaded shipments by day and OD pair,
- empty repositioning by day and OD pair,
- cargo backlog at the start of each day.

This makes the result much easier to interpret.


In [9]:

# -------------------------------------------------------------
# Turn the optimized vector back into named solution values
# -------------------------------------------------------------
solution = {var: result.x[var_index[var]] for var in variables}

def to_int(x):
    # The LP solution comes back exactly integral here,
    # but we round to protect against tiny numerical noise.
    return int(round(float(x)))

avail_df = pd.DataFrame(
    [{i: to_int(solution[("avail", t, i)]) for i in airports} for t in days],
    index=days
)

stay_df = pd.DataFrame(
    [{i: to_int(solution[("stay", t, i)]) for i in airports} for t in days],
    index=days
)

loaded_df = pd.DataFrame(
    [{f"{o}->{d}": to_int(solution[("loaded", t, o, d)]) for (o, d) in ods} for t in days],
    index=days
)

empty_df = pd.DataFrame(
    [{f"{o}->{d}": to_int(solution[("empty", t, o, d)]) for (o, d) in ods} for t in days],
    index=days
)

backlog_df = pd.DataFrame(
    [{f"{o}->{d}": to_int(solution[("backlog", t, o, d)]) for (o, d) in ods} for t in days],
    index=days
)

print("Aircraft available at the beginning of each day")
display(avail_df)

print("Aircraft that stay overnight after each day")
display(stay_df)

print("Loaded cargo movements")
display(loaded_df)

print("Empty repositioning movements")
display(empty_df)

print("Backlog cargo at the beginning of each day")
display(backlog_df)


Aircraft available at the beginning of each day


Aircraft available at the beginning of each day


,A,B,C
Mon,250,510,440
Tue,360,600,240
Wed,205,490,505
Thu,545,415,240
Fri,160,600,440


Aircraft that stay overnight after each day


,A,B,C
Mon,0,0,0
Tue,20,0,0
Wed,55,15,165
Thu,95,0,0
Fri,0,0,0


...

,A->B,A->C,B->A,B->C,C->A,C->B
Mon,200,50,25,25,40,400
Tue,290,50,25,25,40,200
Wed,100,50,25,25,40,300
Thu,400,50,25,25,40,200
Fri,110,50,25,25,40,400


Empty repositioning movements


,A->B,A->C,B->A,B->C,C->A,C->B
Mon,0,0,295,165,0,0
Tue,0,0,120,430,0,0
Wed,0,0,425,0,0,0
Thu,0,0,0,365,0,0
Fri,0,0,185,365,0,0


Backlog cargo at the beginning of each day


,A->B,A->C,B->A,B->C,C->A,C->B
Mon,190,0,0,0,0,0
Tue,90,0,0,0,0,0
Wed,0,0,0,0,0,0
Thu,0,0,0,0,0,0
Fri,0,0,0,0,0,0



## 8. Cost breakdown

Now we separate the objective value into:

- total empty repositioning cost,
- total cargo holding cost,
- total weekly cost.

This is useful because it tells us **where the cost is really coming from**.


In [10]:

# -------------------------------------------------------------
# Cost breakdown
# -------------------------------------------------------------
total_reposition_cost = 0
for t in days:
    for (o, d) in ods:
        total_reposition_cost += reposition_cost[(o, d)] * solution[("empty", t, o, d)]

total_holding_cost = 0
for t in days:
    for (o, d) in ods:
        total_holding_cost += holding_cost_per_day * backlog_holding_weight[t] * solution[("backlog", t, o, d)]

cost_breakdown = pd.Series({
    "Empty repositioning cost": total_reposition_cost,
    "Cargo holding cost": total_holding_cost,
    "Total weekly cost": total_reposition_cost + total_holding_cost,
})

display(cost_breakdown.to_frame("Cost"))


,Cost
Empty repositioning cost,"15,125.00"
Cargo holding cost,"6,600.00"
Total weekly cost,"21,725.00"



## 9. Sanity checks

A good optimization notebook should not just trust the solver output blindly.

We verify three important properties:

1. the fleet size is 1200 each day,
2. all variables are nonnegative,
3. the solution is integral in the base case.

The third point is nice because the problem data are all integer-valued and the solver returned an integer-valued optimum even though we solved the LP in continuous form.


In [11]:

# -------------------------------------------------------------
# Sanity checks
# -------------------------------------------------------------
daily_fleet_totals = avail_df.sum(axis=1)

max_negative = min(result.x)
max_integrality_error = max(abs(v - round(v)) for v in result.x)

print("Daily fleet totals:")
display(daily_fleet_totals.to_frame("Total aircraft"))

print(f"Smallest variable value returned by solver: {max_negative:.12f}")
print(f"Maximum integrality error: {max_integrality_error:.12f}")


Daily fleet totals:


Daily fleet totals:


,Total aircraft
Mon,1200
Tue,1200
Wed,1200
Thu,1200
Fri,1200


Daily fleet totals:


,Total aircraft
Mon,1200
Tue,1200
Wed,1200
Thu,1200
Fri,1200


Smallest variable value returned by solver: 0.000000000000
Maximum integrality error: 0.000000000000



## 10. Interpreting the base-case optimal solution

A few immediate observations from the optimal plan:

- The optimization chooses the Monday morning fleet distribution **endogenously**.
- The only persistent cargo backlog in the base case is on **A→B**, and it disappears by midweek.
- The model uses substantial **empty repositioning from B back to A and C**.
- That makes operational sense:
  - there is a lot of **C→B** cargo,
  - so many aircraft flow into **B loaded**,
  - then the system must reposition aircraft out of **B** to restore balance for the next days and for the repeating weekly cycle.

So the model is effectively telling us that **B becomes a sink for loaded aircraft**, and empty repositioning is the price the company pays to recover a balanced fleet.


In [12]:

# Route-level empty repositioning totals and costs
route_summary = pd.DataFrame({
    "Total empty flights": empty_df.sum(axis=0),
})
route_summary["Unit reposition cost"] = [reposition_cost[tuple(route.split("->"))] for route in route_summary.index]
route_summary["Total reposition cost"] = route_summary["Total empty flights"] * route_summary["Unit reposition cost"]

display(route_summary)


,Total empty flights,Unit reposition cost,Total reposition cost
A->B,0,7,0
A->C,0,3,0
B->A,1025,7,7175
B->C,1325,6,7950
C->A,0,3,0
C->B,0,6,0



## 11. Managerial insight study 1: What is the benefit of increasing fleet size?

The project explicitly asks for managerial implications, and one natural question is:

> How much would it help if Express Air had more than 1200 aircraft available?

To answer that, we resolve the optimization problem for different fleet sizes and compare the optimal weekly cost.

This is a very direct way to quantify the value of additional capacity.


In [13]:

# -------------------------------------------------------------
# Helper function to solve the same model under different scenarios
# -------------------------------------------------------------
def solve_express_air(total_fleet=1200, demand_override=None):
    """Solve the weekly cargo LP for a given fleet size and optional demand modifications."""

    # Start from the base demand table
    local_demand = dict(demand)
    if demand_override is not None:
        local_demand.update(demand_override)

    A_eq_local, b_eq_local = [], []
    A_ub_local, b_ub_local = [], []

    def add_eq_local(coeffs, rhs):
        row = np.zeros(n_vars)
        for var, val in coeffs.items():
            row[var_index[var]] = val
        A_eq_local.append(row)
        b_eq_local.append(rhs)

    def add_ub_local(coeffs, rhs):
        row = np.zeros(n_vars)
        for var, val in coeffs.items():
            row[var_index[var]] = val
        A_ub_local.append(row)
        b_ub_local.append(rhs)

    # Aircraft usage
    for t in days:
        for i in airports:
            coeffs = {
                ("avail", t, i): 1,
                ("stay", t, i): -1,
            }
            for j in airports:
                if j != i:
                    coeffs[("loaded", t, i, j)] = -1
                    coeffs[("empty",  t, i, j)] = -1
            add_eq_local(coeffs, 0)

    # Aircraft transitions
    for t in days:
        t_next = next_day[t]
        for i in airports:
            coeffs = {
                ("avail", t_next, i): 1,
                ("stay",  t, i): -1,
            }
            for j in airports:
                if j != i:
                    coeffs[("loaded", t, j, i)] = -1
                    coeffs[("empty",  t, j, i)] = -1
            add_eq_local(coeffs, 0)

    # Fleet total
    for t in days:
        coeffs = {("avail", t, i): 1 for i in airports}
        add_eq_local(coeffs, total_fleet)

    # Cargo balances and cargo feasibility
    for t in days:
        t_next = next_day[t]
        for (o, d) in ods:
            add_eq_local(
                {
                    ("backlog", t_next, o, d):  1,
                    ("backlog", t,      o, d): -1,
                    ("loaded",  t,      o, d):  1,
                },
                local_demand[(t, o, d)]
            )
            add_ub_local(
                {
                    ("loaded",  t, o, d):  1,
                    ("backlog", t, o, d): -1,
                },
                local_demand[(t, o, d)]
            )

    res = linprog(
        c=c,
        A_ub=np.array(A_ub_local),
        b_ub=np.array(b_ub_local),
        A_eq=np.array(A_eq_local),
        b_eq=np.array(b_eq_local),
        bounds=[(0, None)] * n_vars,
        method="highs"
    )
    return res


In [14]:

# -------------------------------------------------------------
# Fleet-size sensitivity analysis
# -------------------------------------------------------------
fleet_sizes = list(range(1100, 1501, 25))

fleet_sensitivity = []
for fleet in fleet_sizes:
    res_fleet = solve_express_air(total_fleet=fleet)
    fleet_sensitivity.append({
        "Fleet size": fleet,
        "Feasible": res_fleet.success,
        "Optimal weekly cost": np.nan if not res_fleet.success else res_fleet.fun
    })

fleet_sensitivity_df = pd.DataFrame(fleet_sensitivity)
display(fleet_sensitivity_df)


,Fleet size,Feasible,Optimal weekly cost
0,1100,False,NaN
1,1125,False,NaN
2,1150,True,"24,525.00"
3,1175,True,"23,025.00"
4,1200,True,"21,725.00"
5,1225,True,"20,725.00"
6,1250,True,"19,725.00"
7,1275,True,"18,725.00"
8,1300,True,"17,825.00"
9,1325,True,"17,075.00"


In [23]:
# -------------------------------------------------------------
# Adaptive fleet-size search
# Stage 1: increment by 25 until 3 consecutive equal optimal costs
# Stage 2: increment by 10 until 3 consecutive equal optimal costs
# Stage 3: increment by 1 to pin down the absolute minimum
# -------------------------------------------------------------

# This cell depends on model setup objects defined earlier in the notebook.
required_symbols = ["np", "pd", "solve_express_air"]
missing_symbols = [name for name in required_symbols if name not in globals()]

if missing_symbols:
    raise RuntimeError(
        "This cell requires prior setup and helper definitions. "
        f"Missing: {missing_symbols}. "
        "Please run Cells 3 through 21 first, then rerun Cell 22."
    )

start_fleet = 1100
max_fleet = 2500
repeat_needed = 3
tol = 1e-9

def search_with_step(step, start, max_fleet=max_fleet, repeat_needed=3, tol=1e-9):
    """Search fleet sizes with fixed step until repeated optimal cost is seen."""
    rows = []
    streak = 0
    prev_cost = None
    trigger_fleet = None

    for fleet in range(start, max_fleet + 1, step):
        res = solve_express_air(total_fleet=fleet)
        cost = np.nan if not res.success else float(res.fun)

        rows.append({
            "Fleet size": fleet,
            "Step": step,
            "Feasible": res.success,
            "Optimal weekly cost": cost,
        })

        if res.success:
            if prev_cost is not None and abs(cost - prev_cost) <= tol:
                streak += 1
            else:
                streak = 1
            prev_cost = cost

            if streak >= repeat_needed:
                trigger_fleet = fleet
                break
        else:
            # Reset streak on infeasible points.
            streak = 0
            prev_cost = None

    return pd.DataFrame(rows), trigger_fleet

# ---------------- Stage 1: step = 25 ----------------
stage1_df, trigger25 = search_with_step(
    step=25,
    start=start_fleet,
    max_fleet=max_fleet,
    repeat_needed=repeat_needed,
    tol=tol,
 )

if trigger25 is None:
    raise RuntimeError("Stage 1 did not find 3 consecutive repeated optimal costs within max_fleet.")

# ---------------- Stage 2: step = 10 ----------------
# Start near the stage-1 trigger to check if further reduction is possible.
stage2_start = max(start_fleet, trigger25 - 2 * 25)
stage2_df, trigger10 = search_with_step(
    step=10,
    start=stage2_start,
    max_fleet=max_fleet,
    repeat_needed=repeat_needed,
    tol=tol,
 )

if trigger10 is None:
    raise RuntimeError("Stage 2 did not find 3 consecutive repeated optimal costs within max_fleet.")

# ---------------- Stage 3: step = 1 ----------------
# Start near the stage-2 trigger to locate the absolute minimum in this refined region.
stage3_start = max(start_fleet, trigger10 - 2 * 10)
stage3_df, trigger1 = search_with_step(
    step=1,
    start=stage3_start,
    max_fleet=max_fleet,
    repeat_needed=repeat_needed,
    tol=tol,
 )

# Combine all tested points and keep unique fleet-size evaluations.
adaptive_raw_df = pd.concat([stage1_df, stage2_df, stage3_df], ignore_index=True)
adaptive_fleet_df = (
    adaptive_raw_df
    .sort_values(["Fleet size", "Step"])
.drop_duplicates(subset=["Fleet size"], keep="last")
    .sort_values("Fleet size")
    .reset_index(drop=True)
)

# Best result among feasible tested points.
adaptive_feasible_df = adaptive_fleet_df[adaptive_fleet_df["Feasible"]].copy()
best_row = adaptive_feasible_df.loc[adaptive_feasible_df["Optimal weekly cost"].idxmin()]

print("Stage 1 trigger (step=25):", trigger25)
print("Stage 2 trigger (step=10):", trigger10)
print("Stage 3 trigger (step=1):", trigger1)
print("\nAbsolute minimum found in adaptive search:")
print("Fleet size:", int(best_row["Fleet size"]))
print("Optimal weekly cost:", best_row["Optimal weekly cost"])

display(adaptive_fleet_df)

Stage 1 trigger (step=25): 1450
Stage 2 trigger (step=10): 1420
Stage 3 trigger (step=1): 1402

Absolute minimum found in adaptive search:
Fleet size: 1400
Optimal weekly cost: 15125.0


,Fleet size,Step,Feasible,Optimal weekly cost
0,1100,25,False,NaN
1,1125,25,False,NaN
2,1150,25,True,"24,525.00"
3,1175,25,True,"23,025.00"
4,1200,25,True,"21,725.00"
5,1225,25,True,"20,725.00"
6,1250,25,True,"19,725.00"
7,1275,25,True,"18,725.00"
8,1300,25,True,"17,825.00"
9,1325,25,True,"17,075.00"


Increasing the fleet to **1400 planes** is the most optimal fleet size that reduces the weekly costs to a minimum without having excess planes. Having an additional 200 planes will reduce repositiong incured costs, as the demand outbound from airports A and C can be executed by those additional planes without reliance on repositiong the planes that inbound into airport B with no outbound demand.


The lowest size of fleet the company can get away with is 1150 planes, but the cost incurred by the fleet size is **higher**.

In [16]:

# Find the smallest feasible fleet size in the tested range
feasible_only = fleet_sensitivity_df[fleet_sensitivity_df["Feasible"]].copy()

smallest_feasible_fleet = feasible_only["Fleet size"].min()
base_case_cost = feasible_only.loc[feasible_only["Fleet size"] == 1200, "Optimal weekly cost"].iloc[0]
best_tested_cost = feasible_only["Optimal weekly cost"].min()

print("Smallest feasible fleet size found in this search:", smallest_feasible_fleet)
print("Base-case cost at fleet size 1200:", base_case_cost)
print("Best cost in tested range:", best_tested_cost)


Smallest feasible fleet size found in this search: 1150
Base-case cost at fleet size 1200: 21725.0
Best cost in tested range: 15125.0



### Interpretation of the fleet sensitivity

The results show two important things:

1. There is a **minimum fleet size** below which the weekly cycle is infeasible.
2. Increasing fleet size above 1200 continues to reduce cost, mainly because it reduces the need for both:
   - holding cargo on the ground,
   - expensive empty repositioning.

This is a useful managerial result: the model can be used as a **capacity planning tool**, not just as a weekly routing tool.



## 12. Managerial insight study 2: What happens if cargo on a major lane increases?

Another question suggested by the project is the effect of changing cargo levels on specific OD pairs.

The largest and most operationally influential lane in the data is **C→B**. So we study what happens when C→B demand is multiplied by several factors while the fleet stays fixed at 1200.


In [17]:

# -------------------------------------------------------------
# Demand sensitivity for the dominant lane C->B
# -------------------------------------------------------------
multipliers = [0.80, 0.90, 1.00, 1.10, 1.20]

cb_sensitivity = []
for mult in multipliers:
    override = {
        (t, "C", "B"): demand[(t, "C", "B")] * mult
        for t in days
    }
    res_cb = solve_express_air(total_fleet=1200, demand_override=override)
    cb_sensitivity.append({
        "C->B demand multiplier": mult,
        "Feasible": res_cb.success,
        "Optimal weekly cost": np.nan if not res_cb.success else res_cb.fun
    })

cb_sensitivity_df = pd.DataFrame(cb_sensitivity)
display(cb_sensitivity_df)


,C->B demand multiplier,Feasible,Optimal weekly cost
0,0.80,True,"15,425.00"
1,0.90,True,"18,225.00"
2,1.00,True,"21,725.00"
3,1.10,True,"26,225.00"
4,1.20,False,NaN


## <span style="color: red;">You can lowkey skip this with the cell and table output above it. It didn't work as I expected but I still don't want to delete it for some reason lmao</span>
### Interpretation of the C→B demand sensitivity

This study shows that the network is highly sensitive to the **C→B lane**, which is exactly what we would expect from the original data.

Because so much cargo moves into B from C, increasing C→B demand does two things:

- it consumes more aircraft capacity on that lane,
- it creates even stronger pressure to reposition aircraft away from B later.

So the model identifies C→B as a **structurally important lane** in the network.


In [18]:

# -------------------------------------------------------------
# Sensitivity: increase outbound demand from B (B->A and B->C)
# -------------------------------------------------------------
outbound_multipliers = [1.00, 1.25, 1.50, 1.75, 2.00, 2.50, 3.00, 3.50, 4.00, 4.50, 5.00, 5.50, 6.00, 6.50, 7.00, 7.50, 8.00, 8.50, 9.00, 9.50, 10.00, 10.50, 11.00]

outbound_b_sensitivity = []
for mult in outbound_multipliers:
    override = {
        (t, "B", "A"): demand[(t, "B", "A")] * mult
        for t in days
    }
    override.update({
        (t, "B", "C"): demand[(t, "B", "C")] * mult
        for t in days
    })

    res_b_out = solve_express_air(total_fleet=1200, demand_override=override)

    if res_b_out.success:
        sol_local = {var: res_b_out.x[var_index[var]] for var in variables}

        empty_from_b_flights = sum(
            sol_local[("empty", t, "B", j)]
            for t in days
            for j in airports
            if j != "B"
        )
        total_empty_flights = sum(
            sol_local[("empty", t, o, d)]
            for t in days
            for (o, d) in ods
        )
        total_empty_repositioning_cost = sum(
            reposition_cost[(o, d)] * sol_local[("empty", t, o, d)]
            for t in days
            for (o, d) in ods
        )
        total_holding_flights = sum(
            sol_local[("backlog", t, o, d)]
            for t in days
            for (o, d) in ods
        )
        total_holding_cost = sum(
            holding_cost_per_day * backlog_holding_weight[t] * sol_local[("backlog", t, o, d)]
            for t in days
            for (o, d) in ods
        )
    else:
        empty_from_b_flights = np.nan
        total_empty_flights = np.nan
        total_empty_repositioning_cost = np.nan
        total_holding_flights = np.nan
        total_holding_cost = np.nan

    outbound_b_sensitivity.append({
        "Outbound-from-B demand multiplier": mult,
        "Feasible": res_b_out.success,
        "Optimal weekly cost": np.nan if not res_b_out.success else res_b_out.fun,
        "Weekly empty repositioning from B (number of flights)": empty_from_b_flights,
        "Weekly total empty repositioning (number of flights)": total_empty_flights,
        "Weekly total empty repositioning cost": total_empty_repositioning_cost,
        "Weekly total holding (number of flight-loads)": total_holding_flights,
        "Weekly total holding cost": total_holding_cost,
    })

outbound_b_sensitivity_df = pd.DataFrame(outbound_b_sensitivity)
display(outbound_b_sensitivity_df)


,Outbound-from-B demand multiplier,Feasible,Optimal weekly cost,Weekly empty repositioning from B (number of flights),Weekly total empty repositioning (number of flights),Weekly total empty repositioning cost,Weekly total holding (number of flight-loads),Weekly total holding cost
0,1.00,True,"21,725.00","2,350.00","2,350.00","15,125.00",280.00,"6,600.00"
1,1.25,True,"21,318.75","2,287.50","2,287.50","14,718.75",280.00,"6,600.00"
2,1.50,True,"20,912.50","2,225.00","2,225.00","14,312.50",280.00,"6,600.00"
3,1.75,True,"20,506.25","2,162.50","2,162.50","13,906.25",280.00,"6,600.00"
4,2.00,True,"20,100.00","2,100.00","2,100.00","13,500.00",280.00,"6,600.00"
5,2.50,True,"19,287.50","1,975.00","1,975.00","12,687.50",280.00,"6,600.00"
6,3.00,True,"18,475.00","1,850.00","1,850.00","11,875.00",280.00,"6,600.00"
7,3.50,True,"17,662.50","1,725.00","1,725.00","11,062.50",280.00,"6,600.00"
8,4.00,True,"16,850.00","1,600.00","1,600.00","10,250.00",280.00,"6,600.00"
9,4.50,True,"16,037.50","1,475.00","1,475.00","9,437.50",280.00,"6,600.00"


## Interpretation: What Is Optimal When Outbound Cargo Demand Increases?

From the sensitivity table, the best total weekly cost occurs around a **9.5x multiplier** on outbound demand from **B->A** and **B->C** (with fleet fixed at 1200).

### Why this happens
The optimization objective minimizes:

$
\text{Total Weekly Cost} = \text{Empty Repositioning Cost} + \text{Cargo Holding Cost}
$

As outbound demand from B increases:

- **Empty repositioning cost decreases** because fewer aircraft need to be moved empty out of B.
- But after a point, **cargo holding cost increases** because fleet/time constraints become tighter, so backlog pressure grows.

At lower-to-moderate multipliers, the repositioning savings dominate.
At higher multipliers (10.0, 10.5), holding-cost increases become larger than additional repositioning savings, so total cost rises.

That is why the cost-minimizing point is not where repositioning cost is minimum, but where the **sum** of both cost components is minimum.

### Managerial takeaway
The best operating policy is to increase outbound B demand only up to the point where:

- marginal reduction in empty repositioning cost

is approximately equal to

- marginal increase in holding/backlog cost.

In this experiment, that balance occurs around the **9.5x** scenario.

### Practical implication for OD-pair growth
If the airline wants to add cargo on OD pairs, it should prioritize growth that reduces structural aircraft imbalance (like outbound from B), but not so aggressively that backlog/throughput limits dominate and raise total system cost.


In [19]:
# Quick summary stats for the outbound-from-B sensitivity table
summary_cols = [
    "Outbound-from-B demand multiplier",
    "Optimal weekly cost",
    "Weekly total empty repositioning cost",
    "Weekly total holding cost",
    "Weekly empty repositioning from B (number of flights)",
    "Weekly total empty repositioning (number of flights)",
    "Weekly total holding (number of flight-loads)",
]

print("First 5 rows:")
display(outbound_b_sensitivity_df[summary_cols].head())

print("Last 5 rows:")
display(outbound_b_sensitivity_df[summary_cols].tail())

# Work only with feasible scenarios
feasible_df = outbound_b_sensitivity_df[outbound_b_sensitivity_df["Feasible"]].copy()
base_row = feasible_df.iloc[0]

# 1) Compare against the row with minimum total weekly cost
min_total_cost_row = feasible_df.loc[feasible_df["Optimal weekly cost"].idxmin()]
print("\nChange from base row to row with MINIMUM TOTAL WEEKLY COST:")
print("Multiplier at minimum total cost:", min_total_cost_row["Outbound-from-B demand multiplier"])
print("Total cost change:", min_total_cost_row["Optimal weekly cost"] - base_row["Optimal weekly cost"])
print("Empty-repositioning-cost change:", min_total_cost_row["Weekly total empty repositioning cost"] - base_row["Weekly total empty repositioning cost"])
print("Holding-cost change:", min_total_cost_row["Weekly total holding cost"] - base_row["Weekly total holding cost"])
print("Empty-from-B flights change:", min_total_cost_row["Weekly empty repositioning from B (number of flights)"] - base_row["Weekly empty repositioning from B (number of flights)"])
print("Total empty flights change:", min_total_cost_row["Weekly total empty repositioning (number of flights)"] - base_row["Weekly total empty repositioning (number of flights)"])
print("Total holding flight-loads change:", min_total_cost_row["Weekly total holding (number of flight-loads)"] - base_row["Weekly total holding (number of flight-loads)"])

# 2) For each metric, compare against that metric's own minimum row
metric_cols = [
    "Optimal weekly cost",
    "Weekly total empty repositioning cost",
    "Weekly total holding cost",
    "Weekly empty repositioning from B (number of flights)",
    "Weekly total empty repositioning (number of flights)",
    "Weekly total holding (number of flight-loads)",
]

print("\nChange from base row to EACH METRIC'S OWN MINIMUM row:")
for col in metric_cols:
    min_row = feasible_df.loc[feasible_df[col].idxmin()]
    print("\nMetric:", col)
    print("Multiplier at metric minimum:", min_row["Outbound-from-B demand multiplier"])
    print("Total cost change:", min_row["Optimal weekly cost"] - base_row["Optimal weekly cost"])
    print("Empty-repositioning-cost change:", min_row["Weekly total empty repositioning cost"] - base_row["Weekly total empty repositioning cost"])
    print("Holding-cost change:", min_row["Weekly total holding cost"] - base_row["Weekly total holding cost"])
    print("Empty-from-B flights change:", min_row["Weekly empty repositioning from B (number of flights)"] - base_row["Weekly empty repositioning from B (number of flights)"])
    print("Total empty flights change:", min_row["Weekly total empty repositioning (number of flights)"] - base_row["Weekly total empty repositioning (number of flights)"])
    print("Total holding flight-loads change:", min_row["Weekly total holding (number of flight-loads)"] - base_row["Weekly total holding (number of flight-loads)"])

print("\nAll scenarios feasible?", outbound_b_sensitivity_df["Feasible"].all())
print(
    "Total cost monotonically decreasing?",
    outbound_b_sensitivity_df["Optimal weekly cost"].is_monotonic_decreasing,
)
print(
    "Empty-from-B flights monotonically decreasing?",
    outbound_b_sensitivity_df["Weekly empty repositioning from B (number of flights)"].is_monotonic_decreasing,
)
print(
    "Holding-cost monotonically decreasing?",
    outbound_b_sensitivity_df["Weekly total holding cost"].is_monotonic_decreasing,
)


First 5 rows:


,Outbound-from-B demand multiplier,Optimal weekly cost,Weekly total empty repositioning cost,Weekly total holding cost,Weekly empty repositioning from B (number of flights),Weekly total empty repositioning (number of flights),Weekly total holding (number of flight-loads)
0,1.00,"21,725.00","15,125.00","6,600.00","2,350.00","2,350.00",280.00
1,1.25,"21,318.75","14,718.75","6,600.00","2,287.50","2,287.50",280.00
2,1.50,"20,912.50","14,312.50","6,600.00","2,225.00","2,225.00",280.00
3,1.75,"20,506.25","13,906.25","6,600.00","2,162.50","2,162.50",280.00
4,2.00,"20,100.00","13,500.00","6,600.00","2,100.00","2,100.00",280.00


First 5 rows:


,Outbound-from-B demand multiplier,Optimal weekly cost,Weekly total empty repositioning cost,Weekly total holding cost,Weekly empty repositioning from B (number of flights),Weekly total empty repositioning (number of flights),Weekly total holding (number of flight-loads)
0,1.00,"21,725.00","15,125.00","6,600.00","2,350.00","2,350.00",280.00
1,1.25,"21,318.75","14,718.75","6,600.00","2,287.50","2,287.50",280.00
2,1.50,"20,912.50","14,312.50","6,600.00","2,225.00","2,225.00",280.00
3,1.75,"20,506.25","13,906.25","6,600.00","2,162.50","2,162.50",280.00
4,2.00,"20,100.00","13,500.00","6,600.00","2,100.00","2,100.00",280.00


Last 5 rows:


...

First 5 rows:


,Outbound-from-B demand multiplier,Optimal weekly cost,Weekly total empty repositioning cost,Weekly total holding cost,Weekly empty repositioning from B (number of flights),Weekly total empty repositioning (number of flights),Weekly total holding (number of flight-loads)
0,1.00,"21,725.00","15,125.00","6,600.00","2,350.00","2,350.00",280.00
1,1.25,"21,318.75","14,718.75","6,600.00","2,287.50","2,287.50",280.00
2,1.50,"20,912.50","14,312.50","6,600.00","2,225.00","2,225.00",280.00
3,1.75,"20,506.25","13,906.25","6,600.00","2,162.50","2,162.50",280.00
4,2.00,"20,100.00","13,500.00","6,600.00","2,100.00","2,100.00",280.00


Last 5 rows:


,Outbound-from-B demand multiplier,Optimal weekly cost,Weekly total empty repositioning cost,Weekly total holding cost,Weekly empty repositioning from B (number of flights),Weekly total empty repositioning (number of flights),Weekly total holding (number of flight-loads)
18,9.00,"9,025.00","2,125.00","6,900.00",350.00,350.00,310.00
19,9.50,"8,812.50","1,462.50","7,350.00",225.00,262.50,355.00
20,10.00,"9,000.00",900.00,"8,100.00",100.00,200.00,430.00
21,10.50,"10,062.50",587.50,"9,475.00",0.00,162.50,542.50
22,11.00,NaN,NaN,NaN,NaN,NaN,NaN



Change from base row to row with MINIMUM TOTAL WEEKLY COST:
Multiplier at minimum total cost: 9.5
Total cost change: -12912.5
Empty-repositioning-cost change: -13662.5
Holding-cost change: 750.0
Empty-from-B flights change: -2125.0
Total empty flights change: -2087.5
Total holding flight-loads change: 75.0

Change from base row to EACH METRIC'S OWN MINIMUM row:

Metric: Optimal weekly cost
Multiplier at metric minimum: 9.5
Total cost change: -12912.5
Empty-repositioning-cost change: -13662.5
Holding-cost change: 750.0
Empty-from-B flights change: -2125.0
Total empty flights change: -2087.5
Total holding flight-loads change: 75.0

Metric: Weekly total empty repositioning cost
Multiplier at metric minimum: 10.5
Total cost change: -11662.5
Empty-repositioning-cost change: -14537.5
Holding-cost change: 2875.0
Empty-from-B flights change: -2350.0
Total empty flights change: -2187.5
Total holding flight-loads change: 262.5

Metric: Weekly total holding cost
Multiplier at metric minimum: 1.0

## 12B. Joint sensitivity: separate multipliers for B->A and B->C

Instead of forcing one common outbound multiplier from B, this experiment allows:

- one multiplier for **B->A**, and
- a different multiplier for **B->C**.

We test all combinations in a grid and then identify which feasible combination gives the lowest total weekly cost.

In [20]:
# -------------------------------------------------------------
# Joint grid search: separate multipliers for B->A and B->C
# Multipliers are generated from 1.0 in 0.5 steps until first
# infeasible single-lane test for each OD pair.
# -------------------------------------------------------------
step = 0.5


def find_max_feasible_multiplier(origin, dest, step=0.5):
    """Increase one OD multiplier from 1.0 by `step` until first infeasible."""
    current = 1.0

    while True:
        candidate = round(current + step, 10)
        override = {
            (t, origin, dest): demand[(t, origin, dest)] * candidate
            for t in days
        }
        res = solve_express_air(total_fleet=1200, demand_override=override)

        if not res.success:
            return current, candidate

        current = candidate


ba_max_feasible, ba_first_infeasible = find_max_feasible_multiplier("B", "A", step=step)
bc_max_feasible, bc_first_infeasible = find_max_feasible_multiplier("B", "C", step=step)

ba_multipliers = np.arange(1.0, ba_max_feasible + step / 2, step)
bc_multipliers = np.arange(1.0, bc_max_feasible + step / 2, step)

print("B->A max feasible multiplier (single-lane test):", ba_max_feasible)
print("B->A first infeasible multiplier (single-lane test):", ba_first_infeasible)
print("B->C max feasible multiplier (single-lane test):", bc_max_feasible)
print("B->C first infeasible multiplier (single-lane test):", bc_first_infeasible)

joint_results = []
for m_ba in ba_multipliers:
    for m_bc in bc_multipliers:
        override = {
            (t, "B", "A"): demand[(t, "B", "A")] * m_ba
            for t in days
        }
        override.update({
            (t, "B", "C"): demand[(t, "B", "C")] * m_bc
            for t in days
        })

        res_joint = solve_express_air(total_fleet=1200, demand_override=override)

        if res_joint.success:
            sol_joint = {var: res_joint.x[var_index[var]] for var in variables}

            total_empty_repositioning_cost = sum(
                reposition_cost[(o, d)] * sol_joint[("empty", t, o, d)]
                for t in days
                for (o, d) in ods
            )
            total_holding_cost = sum(
                holding_cost_per_day * backlog_holding_weight[t] * sol_joint[("backlog", t, o, d)]
                for t in days
                for (o, d) in ods
            )
            total_empty_flights = sum(
                sol_joint[("empty", t, o, d)]
                for t in days
                for (o, d) in ods
            )
            total_holding_flights = sum(
                sol_joint[("backlog", t, o, d)]
                for t in days
                for (o, d) in ods
            )
        else:
            total_empty_repositioning_cost = np.nan
            total_holding_cost = np.nan
            total_empty_flights = np.nan
            total_holding_flights = np.nan

        joint_results.append({
            "B->A multiplier": m_ba,
            "B->C multiplier": m_bc,
            "Feasible": res_joint.success,
            "Optimal weekly cost": np.nan if not res_joint.success else res_joint.fun,
            "Weekly total empty repositioning cost": total_empty_repositioning_cost,
            "Weekly total holding cost": total_holding_cost,
            "Weekly total empty repositioning (number of flights)": total_empty_flights,
            "Weekly total holding (number of flight-loads)": total_holding_flights,
        })

joint_sensitivity_df = pd.DataFrame(joint_results)

# Best feasible combinations by total weekly cost
joint_feasible = joint_sensitivity_df[joint_sensitivity_df["Feasible"]].copy()
joint_best10 = joint_feasible.nsmallest(20, "Optimal weekly cost")

print("Total combinations tested:", len(joint_sensitivity_df))
print("Feasible combinations:", len(joint_feasible))
print("Top 20 feasible combinations by minimum total weekly cost:")
display(joint_best10)

if not joint_feasible.empty:
    best_joint = joint_best10.iloc[0]
    print("\nBest feasible combination found:")
    print("B->A multiplier:", best_joint["B->A multiplier"])
    print("B->C multiplier:", best_joint["B->C multiplier"])
    print("Optimal weekly cost:", best_joint["Optimal weekly cost"])
    print("Weekly total empty repositioning cost:", best_joint["Weekly total empty repositioning cost"])
    print("Weekly total holding cost:", best_joint["Weekly total holding cost"])


B->A max feasible multiplier (single-lane test): 12.0
B->A first infeasible multiplier (single-lane test): 12.5
B->C max feasible multiplier (single-lane test): 14.0
B->C first infeasible multiplier (single-lane test): 14.5
Total combinations tested: 621
Feasible combinations: 576
Top 20 feasible combinations by minimum total weekly cost:


B->A max feasible multiplier (single-lane test): 12.0
B->A first infeasible multiplier (single-lane test): 12.5
B->C max feasible multiplier (single-lane test): 14.0
B->C first infeasible multiplier (single-lane test): 14.5
Total combinations tested: 621
Feasible combinations: 576
Top 20 feasible combinations by minimum total weekly cost:


,B->A multiplier,B->C multiplier,Feasible,Optimal weekly cost,Weekly total empty repositioning cost,Weekly total holding cost,Weekly total empty repositioning (number of flights),Weekly total holding (number of flight-loads)
426,8.50,11.50,True,"8,212.50",687.50,"7,525.00",100.00,372.50
452,9.00,11.00,True,"8,275.00",625.00,"7,650.00",100.00,385.00
425,8.50,11.00,True,"8,337.50","1,062.50","7,275.00",162.50,347.50
451,9.00,10.50,True,"8,400.00","1,000.00","7,400.00",162.50,360.00
479,9.50,11.00,True,"8,437.50",337.50,"8,100.00",75.00,430.00
400,8.00,12.00,True,"8,450.00",850.00,"7,600.00",150.00,380.00
424,8.50,10.50,True,"8,462.50","1,437.50","7,025.00",225.00,322.50
453,9.00,11.50,True,"8,475.00",250.00,"8,225.00",37.50,417.50
399,8.00,11.50,True,"8,485.00","1,135.00","7,350.00",167.50,355.00
450,9.00,10.00,True,"8,525.00","1,375.00","7,150.00",225.00,335.00



Best feasible combination found:
B->A multiplier: 8.5
B->C multiplier: 11.5
Optimal weekly cost: 8212.5
Weekly total empty repositioning cost: 687.5
Weekly total holding cost: 7525.0


## 12C. Interpretation of the Joint B->A and B->C Grid Search

This joint sensitivity analysis allows different multipliers for outbound demand on B->A and B->C, then tests all combinations in a 0.5-step grid.

### What changed in this version

- The multiplier ranges are now generated automatically.
- Starting from 1.0, each lane is increased by 0.5 until the first infeasible single-lane case is found.
- This extends the tested space beyond the earlier fixed upper bound.

In this run:

- B->A max feasible (single-lane test): 12.0; first infeasible: 12.5.
- B->C max feasible (single-lane test): 14.0; first infeasible: 14.5.

### Best feasible combination found

- B->A multiplier: 8.5
- B->C multiplier: 11.5
- Optimal weekly cost: 8212.5
- Weekly total empty repositioning cost: 687.5
- Weekly total holding cost: 7525.0

### Why this is useful

The objective is:

$
\text{Total Weekly Cost} = \text{Empty Repositioning Cost} + \text{Holding Cost}
$

As outbound demand from B increases:

- empty repositioning pressure usually falls,
- but holding/backlog pressure can rise if throughput becomes tight.

So the best policy is the combination that minimizes the sum of both cost components, not either component alone.

### Managerial recommendation

- Prefer asymmetric OD growth targets (different multipliers for B->A and B->C) instead of a forced common multiplier.
- Use approximately (B->A, B->C) = (8.5, 11.5) as the current best benchmark from this expanded grid.
- If needed, run a finer local search around this point to confirm whether an even lower-cost nearby solution exists.

In [21]:

## -------------------------------------------------------------
# Sensitivity: increase outbound demand from B (B->A and B->C)
# Dynamically increase multiplier until first infeasible case
# -------------------------------------------------------------
multiplier_step = 0.5
max_iterations = 200  # safety cap to avoid infinite loops

outbound_b_sensitivity = []
current_mult = 1.0

for _ in range(max_iterations):
    mult = round(current_mult, 10)

    override = {
        (t, "B", "A"): demand[(t, "B", "A")] * mult
        for t in days
    }
    override.update({
        (t, "B", "C"): demand[(t, "B", "C")] * mult
        for t in days
    })

    res_b_out = solve_express_air(total_fleet=1400, demand_override=override)

    if res_b_out.success:
        sol_local = {var: res_b_out.x[var_index[var]] for var in variables}

        empty_from_b_flights = sum(
            sol_local[("empty", t, "B", j)]
            for t in days
            for j in airports
            if j != "B"
        )
        total_empty_flights = sum(
            sol_local[("empty", t, o, d)]
            for t in days
            for (o, d) in ods
        )
        total_empty_repositioning_cost = sum(
            reposition_cost[(o, d)] * sol_local[("empty", t, o, d)]
            for t in days
            for (o, d) in ods
        )
        total_holding_flights = sum(
            sol_local[("backlog", t, o, d)]
            for t in days
            for (o, d) in ods
        )
        total_holding_cost = sum(
            holding_cost_per_day * backlog_holding_weight[t] * sol_local[("backlog", t, o, d)]
            for t in days
            for (o, d) in ods
        )
    else:
        empty_from_b_flights = np.nan
        total_empty_flights = np.nan
        total_empty_repositioning_cost = np.nan
        total_holding_flights = np.nan
        total_holding_cost = np.nan

    outbound_b_sensitivity.append({
        "Outbound-from-B demand multiplier": mult,
        "Feasible": res_b_out.success,
        "Optimal weekly cost": np.nan if not res_b_out.success else res_b_out.fun,
        "Weekly empty repositioning from B (number of flights)": empty_from_b_flights,
        "Weekly total empty repositioning (number of flights)": total_empty_flights,
        "Weekly total empty repositioning cost": total_empty_repositioning_cost,
        "Weekly total holding (number of flight-loads)": total_holding_flights,
        "Weekly total holding cost": total_holding_cost,
    })

    if not res_b_out.success:
        break

    current_mult += multiplier_step

outbound_b_sensitivity_df = pd.DataFrame(outbound_b_sensitivity)

if outbound_b_sensitivity_df["Feasible"].any():
    last_feasible_mult = outbound_b_sensitivity_df.loc[
        outbound_b_sensitivity_df["Feasible"],
        "Outbound-from-B demand multiplier"
    ].max()
    print("Last feasible multiplier:", last_feasible_mult)

if (~outbound_b_sensitivity_df["Feasible"]).any():
    first_infeasible_mult = outbound_b_sensitivity_df.loc[
        ~outbound_b_sensitivity_df["Feasible"],
        "Outbound-from-B demand multiplier"
    ].min()
    print("First infeasible multiplier:", first_infeasible_mult)

display(outbound_b_sensitivity_df)


Last feasible multiplier: 13.0
First infeasible multiplier: 13.5


,Outbound-from-B demand multiplier,Feasible,Optimal weekly cost,Weekly empty repositioning from B (number of flights),Weekly total empty repositioning (number of flights),Weekly total empty repositioning cost,Weekly total holding (number of flight-loads),Weekly total holding cost
0,1.00,True,"15,125.00","2,350.00","2,350.00","15,125.00",0.00,0.00
1,1.50,True,"14,312.50","2,225.00","2,225.00","14,312.50",0.00,0.00
2,2.00,True,"13,500.00","2,100.00","2,100.00","13,500.00",0.00,0.00
3,2.50,True,"12,687.50","1,975.00","1,975.00","12,687.50",0.00,0.00
4,3.00,True,"11,875.00","1,850.00","1,850.00","11,875.00",0.00,0.00
5,3.50,True,"11,062.50","1,725.00","1,725.00","11,062.50",0.00,0.00
6,4.00,True,"10,250.00","1,600.00","1,600.00","10,250.00",0.00,0.00
7,4.50,True,"9,437.50","1,475.00","1,475.00","9,437.50",0.00,0.00
8,5.00,True,"8,625.00","1,350.00","1,350.00","8,625.00",0.00,0.00
9,5.50,True,"7,812.50","1,225.00","1,225.00","7,812.50",0.00,0.00


## 12D. Optimality Interpretation for Fleet Size 1400

Using the same outbound-demand experiment with fleet fixed at **1400**, optimality is defined as:

$
\min \; \text{Total Weekly Cost} = \text{Empty Repositioning Cost} + \text{Holding Cost}
$

subject to feasibility of the LP constraints.

### What this means in this table

- The **optimal multiplier** is the feasible row with the smallest value in `Optimal weekly cost`.
- Because this cell now increases the multiplier dynamically until the first infeasible case, the search identifies both:
  - the highest feasible multiplier tested, and
  - the first infeasible multiplier boundary.

For the current run in this notebook, the boundary is:

- Last feasible multiplier: **13.0**
- First infeasible multiplier: **13.5**

So, with fleet size 1400, the best operating policy in this experiment is the feasible multiplier within this tested range that gives the minimum total weekly cost (not necessarily the one with minimum repositioning cost alone). In this case, the most optimal outbound increase multiplier from airport B is **10x** which gives us a total weekly operating cost of **$1200**

### Managerial takeaway

At 1400 aircraft, the system can absorb substantially more outbound demand from B before becoming infeasible. This increases the feasible planning range and typically improves the cost tradeoff between empty repositioning and backlog holding, but the optimal point should still be chosen by **minimum total cost among feasible scenarios**.

In [22]:
# -------------------------------------------------------------
# Joint grid search: separate multipliers for B->A and B->C
# Multipliers are generated from 1.0 in 0.5 steps until first
# infeasible single-lane test for each OD pair.
# -------------------------------------------------------------
step = 0.5


def find_max_feasible_multiplier(origin, dest, step=0.5):
    """Increase one OD multiplier from 1.0 by `step` until first infeasible."""
    current = 1.0

    while True:
        candidate = round(current + step, 10)
        override = {
            (t, origin, dest): demand[(t, origin, dest)] * candidate
            for t in days
        }
        res = solve_express_air(total_fleet=1400, demand_override=override)

        if not res.success:
            return current, candidate

        current = candidate


ba_max_feasible, ba_first_infeasible = find_max_feasible_multiplier("B", "A", step=step)
bc_max_feasible, bc_first_infeasible = find_max_feasible_multiplier("B", "C", step=step)

ba_multipliers = np.arange(1.0, ba_max_feasible + step / 2, step)
bc_multipliers = np.arange(1.0, bc_max_feasible + step / 2, step)

print("B->A max feasible multiplier (single-lane test):", ba_max_feasible)
print("B->A first infeasible multiplier (single-lane test):", ba_first_infeasible)
print("B->C max feasible multiplier (single-lane test):", bc_max_feasible)
print("B->C first infeasible multiplier (single-lane test):", bc_first_infeasible)

joint_results = []
for m_ba in ba_multipliers:
    for m_bc in bc_multipliers:
        override = {
            (t, "B", "A"): demand[(t, "B", "A")] * m_ba
            for t in days
        }
        override.update({
            (t, "B", "C"): demand[(t, "B", "C")] * m_bc
            for t in days
        })

        res_joint = solve_express_air(total_fleet=1400, demand_override=override)

        if res_joint.success:
            sol_joint = {var: res_joint.x[var_index[var]] for var in variables}

            total_empty_repositioning_cost = sum(
                reposition_cost[(o, d)] * sol_joint[("empty", t, o, d)]
                for t in days
                for (o, d) in ods
            )
            total_holding_cost = sum(
                holding_cost_per_day * backlog_holding_weight[t] * sol_joint[("backlog", t, o, d)]
                for t in days
                for (o, d) in ods
            )
            total_empty_flights = sum(
                sol_joint[("empty", t, o, d)]
                for t in days
                for (o, d) in ods
            )
            total_holding_flights = sum(
                sol_joint[("backlog", t, o, d)]
                for t in days
                for (o, d) in ods
            )
        else:
            total_empty_repositioning_cost = np.nan
            total_holding_cost = np.nan
            total_empty_flights = np.nan
            total_holding_flights = np.nan

        joint_results.append({
            "B->A multiplier": m_ba,
            "B->C multiplier": m_bc,
            "Feasible": res_joint.success,
            "Optimal weekly cost": np.nan if not res_joint.success else res_joint.fun,
            "Weekly total empty repositioning cost": total_empty_repositioning_cost,
            "Weekly total holding cost": total_holding_cost,
            "Weekly total empty repositioning (number of flights)": total_empty_flights,
            "Weekly total holding (number of flight-loads)": total_holding_flights,
        })

joint_sensitivity_df = pd.DataFrame(joint_results)

# Best feasible combinations by total weekly cost
joint_feasible = joint_sensitivity_df[joint_sensitivity_df["Feasible"]].copy()
joint_best10 = joint_feasible.nsmallest(20, "Optimal weekly cost")

print("Total combinations tested:", len(joint_sensitivity_df))
print("Feasible combinations:", len(joint_feasible))
print("Top 20 feasible combinations by minimum total weekly cost:")
display(joint_best10)

if not joint_feasible.empty:
    best_joint = joint_best10.iloc[0]
    print("\nBest feasible combination found:")
    print("B->A multiplier:", best_joint["B->A multiplier"])
    print("B->C multiplier:", best_joint["B->C multiplier"])
    print("Optimal weekly cost:", best_joint["Optimal weekly cost"])
    print("Weekly total empty repositioning cost:", best_joint["Weekly total empty repositioning cost"])
    print("Weekly total holding cost:", best_joint["Weekly total holding cost"])


B->A max feasible multiplier (single-lane test): 19.5
B->A first infeasible multiplier (single-lane test): 20.0
B->C max feasible multiplier (single-lane test): 21.0
B->C first infeasible multiplier (single-lane test): 21.5
Total combinations tested: 1558
Feasible combinations: 1032
Top 20 feasible combinations by minimum total weekly cost:


B->A max feasible multiplier (single-lane test): 19.5
B->A first infeasible multiplier (single-lane test): 20.0
B->C max feasible multiplier (single-lane test): 21.0
B->C first infeasible multiplier (single-lane test): 21.5
Total combinations tested: 1558
Feasible combinations: 1032
Top 20 feasible combinations by minimum total weekly cost:


,B->A multiplier,B->C multiplier,Feasible,Optimal weekly cost,Weekly total empty repositioning cost,Weekly total holding cost,Weekly total empty repositioning (number of flights),Weekly total holding (number of flight-loads)
676,9.00,11.00,True,725.00,675.00,50.00,125.00,5.00
636,8.50,11.50,True,847.50,847.50,0.00,180.00,0.00
677,9.00,11.50,True,850.00,300.00,550.00,62.50,55.00
716,9.50,10.50,True,887.50,712.50,175.00,137.50,17.50
637,8.50,12.00,True,912.50,487.50,425.00,125.00,42.50
597,8.00,12.50,True,975.00,675.00,300.00,187.50,30.00
596,8.00,12.00,True,"1,010.00","1,010.00",0.00,230.00,0.00
717,9.50,11.00,True,"1,012.50",712.50,300.00,112.50,30.00
678,9.00,12.00,True,"1,025.00",225.00,800.00,50.00,80.00
557,7.50,13.00,True,"1,037.50",862.50,175.00,250.00,17.50



Best feasible combination found:
B->A multiplier: 9.0
B->C multiplier: 11.0
Optimal weekly cost: 725.0
Weekly total empty repositioning cost: 675.0
Weekly total holding cost: 50.0


## 12E. Combined-Strategy Conclusion (Fleet + Outbound Demand from B)

Your hypothesis is directionally correct **for this model and the tested ranges**.

In this notebook, the strongest cost outcome is achieved when managers combine:

- an increased fleet (the best tested fleet level from your fleet-size study), and
- outbound demand growth from airport B (tuned to the best feasible multiplier pattern from your sensitivity tests).

### Why this conclusion is valid here

The objective is
$$
\min\;\text{Total Weekly Cost} = \text{Empty Repositioning Cost} + \text{Holding Cost},
$$
so improving both aircraft balance and throughput from B attacks the two dominant cost drivers at the same time.

### Highlighted results from your analyses

- **Fleet-size study**: cost improves at the optimal tested fleet level versus the 1200-plane base case.
- **Shared outbound multiplier study (fleet fixed at the optimal level)**: there is a best feasible multiplier before backlog pressure starts to dominate.
- **Joint B->A and B->C study**: allowing asymmetric multipliers performs better than forcing a single common multiplier.

### Managerial recommendation

Yes, under this LP formulation, this combined strategy is the best policy among the scenarios you tested. Managers should implement the fleet increase and outbound-demand targeting together, and then re-optimize periodically as demand patterns change.


## 13. Final conclusions for Part 3

### Optimization formulation
This problem is formulated as a linear program with:

- loaded movement decisions,
- empty repositioning decisions,
- overnight stay decisions,
- beginning-of-day aircraft availability states,
- beginning-of-day cargo backlog states.

The objective minimizes weekly empty repositioning cost plus backlog holding cost, subject to aircraft flow balance, cargo flow balance, nonnegativity, fleet-size limits, and a weekly cyclic condition (Friday to Monday).

### Base-case operational pattern (fleet = 1200)
The base model produces a feasible repeatable weekly plan and confirms a clear structural pattern:

- heavy inbound loaded flow to B (especially from C->B) tends to accumulate aircraft at B,
- the network then requires empty repositioning from B to rebalance the fleet for subsequent days and cycle closure.

### Updated sensitivity insights

1. Fleet-size sensitivity with adaptive search
- Using the adaptive step logic (25, then 10, then 1), the best fleet size found is **1400**.
- The corresponding minimum weekly cost in the base-demand fleet search is **15125.0**.
- This confirms that moving from 1200 to 1400 aircraft is the strongest improvement in the tested fleet-sizing space.

2. Outbound-from-B shared-multiplier sensitivity at fleet = 1400
- The best shared multiplier is **10.0x** on both B->A and B->C.
- The minimum weekly cost for this shared-multiplier study is **1200.0**.

3. Joint B->A and B->C sensitivity at fleet = 1400
- Allowing separate multipliers performs better than forcing one common multiplier.
- The best feasible combination found is:
  - B->A multiplier = **9.0**
  - B->C multiplier = **11.0**
  - Optimal weekly cost = **725.0**

### Managerial implications

- Airport B is the main balancing pressure point in this network.
- A two-part policy is supported by the model results:
  - raise fleet capacity to 1400 for the baseline operating environment, and
  - pursue targeted outbound-demand growth from B, preferably asymmetrically by OD pair.
- The best policy is to minimize total system cost (repositioning + holding), not either component alone.
- Re-optimization should be repeated as demand patterns change, since the best multipliers are scenario-dependent.

---

## 14. Short answer summary report

Part 3 was formulated and solved as a linear program that minimizes weekly empty repositioning plus backlog holding cost under aircraft-flow, cargo-flow, fleet-size, nonnegativity, and weekly-cycle constraints. The updated adaptive fleet search finds **1400** as the best fleet size with base-demand minimum cost **15125.0**. At that fleet level, outbound-demand sensitivity from airport B shows that a shared multiplier can reduce cost substantially (best shared case: **10.0x**, cost **1200.0**), and a joint asymmetric policy performs even better (best tested pair: **B->A = 9.0**, **B->C = 11.0**, cost **725.0**). These results reinforce that managers should combine capacity planning with targeted OD-level demand shaping from airport B.